# Chapter 4 — Using Autograd in PyTorch to Solve a Regression Problem
Autograd tính đạo hàm tự động thế nào, và dùng nó để giải bài toán tối ưu ra sao?


## 1. Autograd trong PyTorch

- Muốn tính được gradient, tensor phải tạo với `requires_grad=True` và kiểu dữ liệu **float**.
- Khi một tensor `requires_grad=True` tham gia các phép toán, PyTorch tự ghi nhớ cách nó được tính ra (computation graph) đằng sau nó.
- Gọi `.backward()` trên một tensor scalar sẽ lan truyền ngược qua graph đó; đạo hàm nằm ở thuộc tính `.grad` của biến đầu vào.
- Ví dụ tối giản: `y = x*x` tại `x=3.6` → đạo hàm đúng là `y' = 2x = 7.2`; `y.backward()` rồi đọc `x.grad` cho đúng kết quả này mà không cần tự viết công thức đạo hàm.

## 2. Polynomial Regression

- Bài toán: có các mẫu `(x, y)` sinh ra từ một đa thức bậc 2 `Ax² + Bx + C` chưa biết hệ số, cần tìm lại A, B, C.
- Cách giải: coi 3 hệ số là một tensor `w` có `requires_grad=True`, dùng optimizer (`NAdam`) chạy vòng lặp gradient descent để giảm dần MSE giữa `y` thực và `y_pred = x @ w`.
- Không cần tự đạo hàm công thức MSE theo `w` — `mse.backward()` tự tính gradient cho cả 3 hệ số cùng lúc.
- 20 mẫu là dư sức để hồi quy chính xác 3 ẩn số (bài toán overdetermined: 20 phương trình, 3 ẩn).

## 3. Math Puzzle

- Autograd không chỉ dùng để huấn luyện neural network — bất kỳ bài toán nào biểu diễn được dưới dạng tối thiểu hoá sai số bình phương đều giải được bằng gradient descent.
- Ví dụ: hệ 4 phương trình tuyến tính với 4 ẩn A, B, C, D. Khởi tạo ngẫu nhiên, tối ưu tổng bình phương sai số của cả 4 phương trình.
- Hệ phương trình này có thể có nhiều nghiệm hợp lệ — gradient descent chỉ tìm ra **một** nghiệm khả dĩ, không phải nghiệm duy nhất hay "đúng nhất".

**Kết luận chính:** cốt lõi của autograd nằm ở chỗ nó tổng quát hơn nhiều so với việc chỉ huấn luyện mạng neural. Bất kỳ đại lượng nào viết được dưới dạng một biểu thức khả vi của các biến `requires_grad=True` đều tối ưu được bằng gradient descent — neural network chỉ là một trường hợp áp dụng cụ thể. Đây chính là cơ chế nền cho mọi training loop PyTorch sẽ gặp ở các chapter sau.


## 4. Vận dụng


**4.1** — Tạo tensor hằng số và kiểm tra shape/dtype

In [1]:
import torch

x = torch.tensor([1, 2, 3])
print(x)
print(x.shape)
print(x.dtype)


tensor([1, 2, 3])
torch.Size([3])
torch.int64


**4.2** — Tạo tensor hỗ trợ tính đạo hàm với requires_grad=True

In [2]:
import torch

x = torch.tensor([1., 2., 3.], requires_grad=True)
print(x)
print(x.shape)
print(x.dtype)


tensor([1., 2., 3.], requires_grad=True)
torch.Size([3])
torch.float32


**4.3** — Tính đạo hàm y=x*x bằng backward()

In [3]:
import torch

x = torch.tensor(3.6, requires_grad=True)
y = x * x
y.backward()
print("x =", x)
print("y =", y)
print("x.grad =", x.grad)


x = tensor(3.6000, requires_grad=True)
y = tensor(12.9600, grad_fn=<MulBackward0>)
x.grad = tensor(7.2000)


**4.4** — Tạo đa thức bậc 2 bằng NumPy poly1d

In [4]:
import numpy as np

polynomial = np.poly1d([1, 2, 3])
print(polynomial)


   2
1 x + 2 x + 3


**4.5** — Dùng đa thức NumPy như một hàm số

In [5]:
import numpy as np

polynomial = np.poly1d([1, 2, 3])
print(polynomial(1.5))


8.25


**4.6** — Sinh mẫu (X, Y) ngẫu nhiên từ đa thức

In [6]:
import numpy as np

polynomial = np.poly1d([1, 2, 3])
N = 20   # number of samples

# Generate random samples roughly between -10 to +10
X = np.random.randn(N,1) * 5
Y = polynomial(X)
print(X)
print(Y)


[[ 4.9336799 ]
 [ 2.35305503]
 [ 9.58282532]
 [-0.36588517]
 [-2.38826915]
 [ 3.08990243]
 [ 0.42843389]
 [ 7.52139645]
 [-5.12976387]
 [-7.65875524]
 [ 1.70706585]
 [-0.25832435]
 [-4.12775488]
 [ 2.83472063]
 [-2.31639813]
 [ 1.71920188]
 [-5.6147048 ]
 [-1.78034513]
 [-2.00077044]
 [-0.64352367]]
[[ 37.20855718]
 [ 13.24297801]
 [113.99619186]
 [  2.40210162]
 [  3.92729123]
 [ 18.72730187]
 [  4.04042339]
 [ 74.6141975 ]
 [ 19.05494963]
 [ 46.33902139]
 [  9.32820549]
 [  2.55008277]
 [ 11.78285057]
 [ 16.70508231]
 [  3.73290402]
 [  9.39405885]
 [ 23.29550041]
 [  2.60893852]
 [  3.00154147]
 [  2.12707538]]


**4.7** — Gradient descent tìm lại hệ số đa thức bằng autograd

In [7]:
import numpy as np
import torch

polynomial = np.poly1d([1, 2, 3])

# Generate random samples roughly between -10 to +10
N = 20   # number of samples
X = np.random.randn(N,1) * 5
Y = polynomial(X)

XX = np.hstack([X*X, X, np.ones_like(X)])

w = torch.randn(3, 1, requires_grad=True)  # the 3 coefficients
x = torch.tensor(XX, dtype=torch.float32)  # input sample
y = torch.tensor(Y, dtype=torch.float32)   # output sample
optimizer = torch.optim.NAdam([w], lr=0.01)
print(w)

for _ in range(1000):
    y_pred = x @ w
    mse = torch.mean(torch.square(y - y_pred))
    optimizer.zero_grad()
    mse.backward()
    optimizer.step()

print(w)


tensor([[ 0.0072],
        [-1.1036],
        [-0.1121]], requires_grad=True)
tensor([[1.0293],
        [1.8094],
        [1.6025]], requires_grad=True)


**4.8** — Chương trình đầy đủ: sinh mẫu + fit đa thức bằng autograd

In [8]:
import numpy as np
import torch

polynomial = np.poly1d([1, 2, 3])
N = 20   # number of samples

# Generate random samples roughly between -10 to +10
X = np.random.randn(N,1) * 5
Y = polynomial(X)

# Prepare input as an array of shape (N,3)
XX = np.hstack([X*X, X, np.ones_like(X)])

# Prepare tensors
w = torch.randn(3, 1, requires_grad=True)  # the 3 coefficients
x = torch.tensor(XX, dtype=torch.float32)  # input sample
y = torch.tensor(Y, dtype=torch.float32)   # output sample
optimizer = torch.optim.NAdam([w], lr=0.01)
print(w)

# Run optimizer
for _ in range(1000):
    optimizer.zero_grad()
    y_pred = x @ w
    mse = torch.mean(torch.square(y - y_pred))
    mse.backward()
    optimizer.step()

print(w)


tensor([[-1.0784],
        [-0.4395],
        [-0.9079]], requires_grad=True)
tensor([[1.0280],
        [2.0742],
        [1.6187]], requires_grad=True)


**4.9** — Giải hệ phương trình 4 ẩn bằng autograd

In [9]:
import random
import torch

A = torch.tensor(random.random(), requires_grad=True)
B = torch.tensor(random.random(), requires_grad=True)
C = torch.tensor(random.random(), requires_grad=True)
D = torch.tensor(random.random(), requires_grad=True)

# Gradient descent loop
EPOCHS = 2000
optimizer = torch.optim.NAdam([A, B, C, D], lr=0.01)
for _ in range(EPOCHS):
    y1 = A + B - 9
    y2 = C - D - 1
    y3 = A + C - 8
    y4 = B - D - 2
    sqerr = y1*y1 + y2*y2 + y3*y3 + y4*y4
    optimizer.zero_grad()
    sqerr.backward()
    optimizer.step()

print(A)
print(B)
print(C)
print(D)


tensor(4.5817, requires_grad=True)
tensor(4.4182, requires_grad=True)
tensor(3.4183, requires_grad=True)
tensor(2.4182, requires_grad=True)


## 5. Kết luận

- Autograd = cơ chế tính đạo hàm tự động của PyTorch, kích hoạt bằng `requires_grad=True`, lấy kết quả qua `.grad` sau khi gọi `.backward()`.
- Autograd tổng quát hơn deep learning: dùng được cho bất kỳ bài toán tối ưu số nào biểu diễn được dưới dạng biểu thức khả vi — hồi quy đa thức, giải hệ phương trình, hay huấn luyện neural network đều cùng một cơ chế.